# Crypto Pipeline — Exploration & Validation
Notebook de validation des données à chaque couche : Bronze → Silver → Gold → Snowflake.

## 0. Setup

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import boto3
from botocore.client import Config
from dotenv import load_dotenv
import os

load_dotenv('../.env')
print('Setup OK')

## 1. Bronze — Vérification JSON brut

In [ ]:
from src.clients.minio_client import get_minio_client
from datetime import datetime, timezone

client = get_minio_client()
today  = datetime.now(timezone.utc)
key    = today.strftime('%Y/%m/%d/raw.json')

response = client.get_object(Bucket='crypto-bronze', Key=key)
payload  = json.loads(response['Body'].read())

print(f"Collected at : {payload['collected_at']}")
print(f"Count        : {payload['count']} cryptos")
print(f"First record : {payload['data'][0]['name']} — ${payload['data'][0]['current_price']}")

## 2. Silver — Vérification Parquet nettoyé

In [ ]:
import io

key      = today.strftime('%Y/%m/%d/clean.parquet')
response = client.get_object(Bucket='crypto-silver', Key=key)
df       = pd.read_parquet(io.BytesIO(response['Body'].read()))

print(f'Shape : {df.shape}')
print(f'Colonnes : {list(df.columns)}')
df.head(5)

In [ ]:
# Vérification qualité Silver
print('Nulls :')
print(df.isnull().sum())
print()
print('Types :')
print(df.dtypes)
print()
print('Stats prix :')
print(df['current_price'].describe())

## 3. Gold — Vérification modèle dimensionnel

In [ ]:
tables = ['fact_crypto_prices', 'dim_crypto', 'dim_date', 'dim_category', 'dim_platform']
gold_dfs = {}

for table in tables:
    key      = today.strftime(f'%Y/%m/%d/{table}.parquet')
    response = client.get_object(Bucket='crypto-gold', Key=key)
    gold_dfs[table] = pd.read_parquet(io.BytesIO(response['Body'].read()))
    print(f'{table:30s} — {len(gold_dfs[table])} rows')

In [ ]:
# Vérification intégrité référentielle
fact = gold_dfs['fact_crypto_prices']
dim_crypto   = gold_dfs['dim_crypto']
dim_date     = gold_dfs['dim_date']
dim_category = gold_dfs['dim_category']
dim_platform = gold_dfs['dim_platform']

print('FK checks:')
print(f"  crypto_key   OK: {fact['crypto_key'].isin(dim_crypto['crypto_key']).all()}")
print(f"  date_key     OK: {fact['date_key'].isin(dim_date['date_key']).all()}")
print(f"  category_key OK: {fact['category_key'].isin(dim_category['category_key']).all()}")
print(f"  platform_key OK: {fact['platform_key'].isin(dim_platform['platform_key']).all()}")

## 4. Snowflake — Vérification données chargées

In [ ]:
from src.clients.snowflake_client import get_snowflake_connection, fetchall

conn = get_snowflake_connection()

for table in ['dim_category', 'dim_platform', 'dim_crypto', 'dim_date', 'fact_crypto_prices']:
    result = fetchall(conn, f'SELECT COUNT(*) AS cnt FROM {table}')
    print(f'{table:30s} — {result[0]["CNT"]} rows')

conn.close()

In [ ]:
# Top 10 cryptos par market cap
conn = get_snowflake_connection()

rows = fetchall(conn, """
    SELECT c.name, c.symbol, f.current_price, f.market_cap, f.market_cap_rank
    FROM fact_crypto_prices f
    JOIN dim_crypto c ON f.crypto_key = c.crypto_key
    ORDER BY f.market_cap_rank ASC
    LIMIT 10
""")

conn.close()
pd.DataFrame(rows)